# Pretrained Networks

Training a large image classifier from scratch needs a huge labelled dataset and a lot of compute. A **pretrained network** lets us skip most of that: it is a model already trained on a large image dataset, whose learned filters we can reuse. We can either use it directly for classification, or adapt it to a new task through **transfer learning**.

This notebook has two parts. First we use a pretrained VGG16 model to classify an image, and see why matching the model's preprocessing matters. Then we use transfer learning to turn that same VGG16 into a small phone-versus-wallet classifier trained on only a handful of images.

## Learning Objectives

At the end of this notebook, you should be able to:

- Load a pretrained Keras model (VGG16) and use it to classify an image.
- Apply the model's required preprocessing and explain why it changes the prediction.
- Build a transfer learning model by freezing a convolutional base and adding a new classification head.
- Train and evaluate that model on a small custom image dataset.

## What is a Pretrained Network?

A pretrained network is a model that was trained on a large-scale image classification task. We can use it as-is for image classification, or as the starting point for transfer learning on a new, related task.

**Benefits:**

- We do not have to train the model from scratch.
- It is easy to incorporate.
- Inference is fast.
- We can reach very good performance.

**Examples:** U-Net (features on medical images), MobileNet, VGG16/19, ResNet, InceptionV3.

Most of the pretrained networks in Keras are trained on the [1000 classes](https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a) of the [ImageNet](https://www.image-net.org/update-mar-11-2021.php) dataset.

In [ ]:
# import libraries
from tensorflow.keras.applications.vgg16 import (
    VGG16,
    decode_predictions,
    preprocess_input,
)
from tensorflow.keras.applications.resnet50 import (
    decode_predictions,
    preprocess_input,
)
from tensorflow.keras.preprocessing import image  # Keras own inbuild image class
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input, Model

import numpy as np
import matplotlib.pyplot as plt

## Classifying an Image with VGG16

Let's load an image that belongs to one of the 1000 ImageNet classes (a few are in the `assets` directory), use the pretrained VGG16 model, and check the classification result.

### Load and Inspect the Image

We read the image as a PIL object and resize it to the shape the pretrained model expects (VGG16 was trained on 224x224 images).

In [ ]:
img = image.load_img("assets/download.jpeg", target_size=(224, 224))

In [ ]:
type(img)

Convert the image to a NumPy array (we can also specify the dtype):

In [ ]:
# convert image to array, can also specify datatype
img = image.img_to_array(img, dtype="uint8")

In [ ]:
img.shape

Plot the image:

In [ ]:
# plot image
plt.imshow(img)

### Load the VGG16 Model

We load VGG16 with its full set of weights. With no arguments, Keras downloads the model trained on ImageNet the first time you run this.

In [ ]:
# initialize the model
model = VGG16()

In [ ]:
# show model summary
model.summary()

The result, an explanation: VGG16 has about **138 million parameters**. That is why training such a network from scratch is expensive, and why reusing the pretrained weights is so valuable.

Check the input shape the model expects:

In [ ]:
# check shape required by model
model.input.shape

#### Expand the Dimensions

The model expects a batch of images, so we add a leading batch dimension to turn the single 224x224x3 image into a batch of one.

In [ ]:
# Reshape to match the input shape required by the model
img = np.expand_dims(img, axis=0)  # or img = img.reshape(1,224,224,3)

In [ ]:
img.shape

Predict the class probabilities, then check the shape of the prediction:

In [ ]:
pred = model.predict(img)

In [ ]:
# shape of pred
pred.shape

Decode the 1000-class probability vector into human-readable labels:

In [ ]:
# decode labels
decode_predictions(pred)

The result, an explanation: the image is a cartoon parrot, but the top prediction is `hair_slide` (about 30%), which is wrong. The pixels were fed in raw, scaled differently from how VGG16 was trained. This is why we need to preprocess the image the same way the model's authors did.

### Preprocess the Image

To get good predictions we must preprocess images exactly as the researchers did when training the model. For VGG16:

> The images are converted from RGB to BGR, then each colour channel is zero-centred with respect to the ImageNet dataset, without scaling.

See the [`preprocess_input` documentation](https://www.tensorflow.org/api_docs/python/tf/keras/applications/vgg16/preprocess_input).

In [ ]:
img = image.load_img("assets/download.jpeg", target_size=(224, 224))
# in the next run try changing the path to assets/object.png and test the prediction of pretrained model
img = np.expand_dims(img, axis=0)  # or img = img.reshape(1,224,224,3)
img = preprocess_input(
    img
)  # preprocess the image in the same method that is used for the pretrained model

The preprocessed image looks distorted when plotted, because the channels have been shifted and reordered:

In [ ]:
plt.imshow(img[0])

#### Predict Again

In [ ]:
pred = model.predict(img)

In [ ]:
# shape of pred
pred.shape

In [ ]:
# decode labels
decode_predictions(pred)

The result, an explanation: after preprocessing, the top prediction is `macaw` (about 31%), which is correct: the image is a cartoon parrot. The only thing that changed was matching the model's preprocessing, and that flipped the prediction from wrong to right.

Now load the `assets/object.png` image and check the result with VGG16. You should get the predicted class `cleaver`. Discuss why the model predicts a phone as a `cleaver`.

## Transfer Learning

### How Transfer Learning Works

> Transfer learning consists of taking features learned on one problem, and using them on a new, similar problem. For example, features from a model that learned to identify racoons may help kick-start a model that identifies tanukis (Japanese racoons).

**Benefits:**

- You reuse a pretrained network.
- It saves a lot of training time.
- It works with very small training datasets.

**Procedure:**

1. Take the weights and architecture of a [pretrained network](https://keras.io/api/applications/).
2. Load the convolutional base (everything except the final dense layers).
3. Freeze the base so its weights stay fixed.
4. Add a fully connected dense layer on top.
5. Add a task-specific dense output layer.
6. Compile and fit the model on your data.

```mermaid
flowchart LR
    A[Input image] --> B[VGG16 convolutional base<br/>frozen, pretrained]
    B --> C[Flatten]
    C --> D[New Dense output layer<br/>trainable, 2 classes]
    D --> E[phone or wallet]
```

Only the new head (in this notebook, the final dense layer) is trained. The frozen base keeps the generic visual features VGG16 already learned from ImageNet.

### Load the Data with ImageDataGenerator

[`ImageDataGenerator`](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator) generates batches of image tensors, with optional real-time data augmentation. It is an efficient way to load data that does not fit in memory, because it streams and trains in small batches from a directory.

In [ ]:
# folder names containing images of the things you want to classify
classes = ["phone", "wallet"]

**Data augmentation** applies random transformations to the training images (flips, rotations, shifts, scaling, and so on). It can address limited or imbalanced data and reduce overfitting by enlarging and varying the training set. Apply it to the training data only.

In [ ]:
# define an image data generator
# Data augmentation: Applies random distortions and transformations to the images (only on your training data!).

data_gen = image.ImageDataGenerator(
    # define the preprocessing function that should be applied to all images
    preprocessing_function=preprocess_input,
    # fill_mode='nearest',
    # rotation_range=20,
    # width_shift_range=0.2,
    # height_shift_range=0.2,
    # horizontal_flip=True,
    # zoom_range=0.2,
    # shear_range=0.2
)

In [ ]:
# -o overwrites without prompting so the cell runs non-interactively
!unzip -o -q data.zip

In [ ]:
# a generator that returns batches of X and y arrays
train_data_gen = data_gen.flow_from_directory(
    directory="data/train",
    class_mode="categorical",
    classes=classes,
    batch_size=150,
    target_size=(224, 224),
)

In [ ]:
val_data_gen = data_gen.flow_from_directory(
    directory="data/validation/",
    class_mode="categorical",
    classes=classes,
    batch_size=150,
    target_size=(224, 224),
)

The result, an explanation: the generator finds 144 training images and 6 validation images across the 2 classes. This is a very small dataset, which is exactly the situation where transfer learning shines.

In [ ]:
train_data_gen.class_indices

### Build the Transfer Learning Model

Instead of building a CNN from scratch, we reuse VGG16's convolutional base and fine-tune it to our dataset by replacing the final layers.

#### 1. Select the Convolutional Base

Loading the model with `include_top=False` drops VGG16's final fully connected layers (its ImageNet classifier), leaving just the convolutional feature extractor so we can attach our own classifier.

In [ ]:
import tensorflow.keras.backend as K

K.clear_session()

# 1. Select the convolutional base / Pretrained network
base_model = VGG16(include_top=False)

In [ ]:
base_model.summary()

The result, an explanation: the convolutional base has about **14.7 million parameters**, down from the 138 million of the full VGG16, because the large dense classifier layers have been removed.

#### 2. Freeze the Weights

We freeze the base so its pretrained weights are not updated during training. This preserves the features VGG16 already learned and means we only train the small new head we add next.

In [ ]:
# 2. Freeze the weights in order to not retrain the loaded pre-trained model
base_model.trainable = False

In [ ]:
base_model.summary()

The result, an explanation: after freezing, all 14.7 million parameters are now non-trainable. The base will act as a fixed feature extractor.

#### 3. Add Your Own Dense Layers

We have only 2 classes to classify (`phone` and `wallet`), so we flatten the base's output and add a single dense layer with 2 units and a softmax activation as the new output.

In [ ]:
# 3. Create your model with pretrained network as base model
inputs = Input(shape=(224, 224, 3))

base = base_model(inputs)

# can also add additional cnn layers if necessary

# dont forget to flatten out before the final layer
flatten = Flatten()(base)

outputs = Dense(2, activation="softmax")(flatten)

model_tf = Model(inputs, outputs)

In [ ]:
model_tf.summary()

The result, an explanation: the full model has about 14.76 million parameters, but only **50,178** of them are trainable (the new dense head). The 14.7 million frozen parameters stay fixed. Training so few parameters is fast and works even with our tiny dataset.

### Compile and Train

We compile with the Adam optimiser and categorical cross-entropy (two one-hot classes), then train for 10 epochs.

In [ ]:
model_tf.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model_tf.fit(
    train_data_gen,
    verbose=2,  # how the training log should get printed
    epochs=10,
    validation_data=val_data_gen,
)

The result, an explanation: training accuracy climbs close to 99%, and validation accuracy reaches about 0.83 (5 of the 6 validation images correct). With only 144 training and 6 validation images the validation score is noisy, but transfer learning still reaches strong accuracy quickly because the frozen VGG16 base already supplies rich visual features.

### Use It to Predict

We reuse the `img` array from earlier (the preprocessed parrot) to show the prediction mechanics. The model now outputs a probability for each of the two trained classes, `phone` and `wallet`.

In [ ]:
model_tf.predict(img)

In [ ]:
plt.imshow(img[0])

In [ ]:
img.shape

Plot the predicted probability for each class:

In [ ]:
plt.bar(x=["phone", "wallet"], height=model_tf.predict(img)[0])

Map the index of the highest probability back to its class name:

In [ ]:
# For remapping the index values of highest prediction probability to its respective class

pred = model_tf.predict(img)

preds_cls = list(train_data_gen.class_indices.keys())[
    list(train_data_gen.class_indices.values()).index(np.argmax(pred, axis=-1))
]
preds_cls

### Save the Model

Save the trained model to disk so it can be reloaded later without retraining.

In [ ]:
import os
os.makedirs("models", exist_ok=True)
model_tf.save("models/wallet_phone.h5")

### Exercise

- Try other pretrained models as the base, such as ResNet50.
- Choose an [image classification dataset](https://www.kaggle.com/datasets?search=image) and build your own CNN (optional): try building it from scratch, then with transfer learning.
- Big CNNs on bigger datasets may exceed your laptop's hardware. Free cloud resources offer GPUs (which speed up training): [Google Colab](https://colab.research.google.com/) and [Kaggle kernels](https://www.kaggle.com/code) (click `New notebook`; a verified phone number is needed for GPU access).

## Appendix: Load All Images into Arrays

This is an alternative to `ImageDataGenerator`: load every image and label directly into NumPy arrays. It is simple and fine for a small dataset that fits in memory, but it does not stream or augment data the way the generator does.

In [ ]:
# Let's explore the data folder
import os

base_path = "data/train/"

# Let's define the classes
classes = os.listdir(base_path)

In [ ]:
for class_ in classes:
    print(class_)

In [ ]:
def load_image(base_path):
    """it loads all the image into X and the classes in y"""
    X_list = []
    y_list = []
    classes = os.listdir(base_path)
    for class_ in classes:
        if class_ != ".DS_Store":
            files = os.listdir(base_path + class_)
            for file in files:
                pic = image.load_img(
                    path=base_path + class_ + "/" + f"{file}", target_size=(224, 224)
                )
                numpy_image = np.array(pic)
                processed_image = preprocess_input(numpy_image)
                X_list.append(processed_image)
                y_list.append(class_)

    X = np.array(X_list)
    y = np.array(y_list)

    return X, y, classes

In [ ]:
X, y, classes = load_image(base_path)

In [ ]:
X.shape

In [ ]:
y

In [ ]:
my_dict = {"wallet": 0, "phone": 1}

In [ ]:
# map strings to binary labels
y = np.vectorize(my_dict.get)(y)
y

## References & Further Reading

- [**Keras Applications (pretrained models)**](https://keras.io/api/applications/): the catalogue of pretrained networks, including VGG16 and ResNet50.
- [**Transfer learning and fine-tuning (Keras guide)**](https://keras.io/guides/transfer_learning/): the official guide to freezing a base and training a new head.
- [**Data augmentation (TensorFlow tutorial)**](https://www.tensorflow.org/tutorials/images/data_augmentation): techniques to enlarge and vary an image dataset.
- [**Transfer learning and fine-tuning (TensorFlow tutorial)**](https://www.tensorflow.org/tutorials/images/transfer_learning): an end-to-end, runnable transfer learning walkthrough on a real image dataset.